# Notebook 05 — Evaluation: PConv vs Vanilla U-Net vs Gated U-Net

**Purpose:** Final test-set evaluation comparing the three trained
models with statistical proof.

**Fairness contract:** Every model is evaluated on the **same**
(image, mask) pairs.  Test masks are pre-generated once with a fixed
seed, cached to disk, and read identically by all three models.

**What this notebook produces:**
1. Per-image PSNR / SSIM / LPIPS for each (model, image, damage, difficulty)
2. Per-cell FID (3 models × 3 difficulties = 9 numbers)
3. Paired Wilcoxon signed-rank tests with Bonferroni correction
4. Bootstrap 95 % CI on per-image effect sizes
5. LaTeX tables (Tables 1 & 2) and Matplotlib figures (Figures 1–3)

**Resume safety:** Heavy compute cells write intermediate artefacts to
Drive (`outputs/eval/`).  Re-running the notebook after a disconnect
skips already-completed work.

## Cell 1 — Setup

Install dependencies, mount Drive, restore project code from Drive cache.

In [ ]:
!pip install -q torch torchvision albumentations pyyaml tqdm pandas \
               matplotlib Pillow opencv-python scikit-learn scipy \
               'torchmetrics[image]' lpips

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, sys
from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive/art_restoration')
DRIVE_CODE  = DRIVE_ROOT / 'code'
PROJECT_DIR = Path('/content/art-restoration')
EVAL_ROOT   = DRIVE_ROOT / 'outputs' / 'eval'
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

if DRIVE_CODE.exists() and (DRIVE_CODE / 'src').exists():
    print(f'Restoring code from Drive cache: {DRIVE_CODE}')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DRIVE_CODE, PROJECT_DIR)
else:
    print('Drive cache empty — cloning private repo from GitHub.')
    import getpass
    token = os.environ.get('GITHUB_TOKEN') or getpass.getpass('GitHub token: ')
    repo_url = f'https://{token}@github.com/orivalo/art_restoration.git'
    !git clone -b phase2-partial-conv {repo_url} {PROJECT_DIR}
    if not (PROJECT_DIR / 'src').exists():
        raise RuntimeError('Clone failed — check your token has `repo` scope.')
    if DRIVE_CODE.exists():
        shutil.rmtree(DRIVE_CODE)
    shutil.copytree(PROJECT_DIR, DRIVE_CODE,
        ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc'))

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f'\nPROJECT_DIR : {PROJECT_DIR}')
print(f'DRIVE_ROOT  : {DRIVE_ROOT}')
print(f'EVAL_ROOT   : {EVAL_ROOT}')
print('Setup complete.')

## Cell 2 — GPU Verification

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required for evaluation (LPIPS + FID are slow on CPU).')
props = torch.cuda.get_device_properties(0)
print(f'GPU      : {props.name}')
print(f'VRAM     : {props.total_memory / 1024**3:.1f} GB')
print(f'PyTorch  : {torch.__version__}')

## Cell 3 — Configs & Checkpoints

Load the three experiment configs and verify each has a trained
checkpoint at `<checkpoint.dir>/best.pth` on Drive.  Stops with a
clear error if any model is still untrained.

In [ ]:
import yaml

MODELS = ['pconv_unet', 'unet_baseline', 'gated_unet']
configs = {}

for arch in MODELS:
    cfg_path = PROJECT_DIR / 'configs' / 'experiment_configs' / f'{arch}.yaml'
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    # Override Drive paths (in case cfg points at /content/drive paths used during training)
    cfg['data']['splits_dir'] = str(DRIVE_ROOT / 'data' / 'splits')
    cfg['checkpoint']['dir']  = str(DRIVE_ROOT / 'checkpoints' / cfg['experiment']['name'])
    configs[arch] = cfg

print('Trained-model status')
print('━' * 60)
missing = []
for arch, cfg in configs.items():
    best = Path(cfg['checkpoint']['dir']) / 'best.pth'
    if best.exists():
        size_mb = best.stat().st_size / 1024**2
        print(f'  {arch:14s} : best.pth ({size_mb:.0f} MB) ✓')
    else:
        print(f'  {arch:14s} : MISSING at {best}  ✗')
        missing.append(arch)

if missing:
    raise RuntimeError(
        f'Cannot evaluate — train these first: {missing}\n'
        f'Run notebook 02 (PConv), 03 (Vanilla), 04 (Gated) before this notebook.'
    )
print('\nAll three models are trained and ready for evaluation.')

## Cell 4 — Pre-compute Test Masks (deterministic)

Generate **one mask per (test image, damage type, difficulty) cell** with
a fixed seed.  Cache as PNG to `outputs/eval/test_masks/`.  All three
models will read these exact same masks.

Idempotent: skips images whose 12 masks already exist on disk.

Output count: `len(test) × 4 damage types × 3 difficulties` masks.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from src.data.mask_generator import MaskGenerator, DIFFICULTY_RANGES

# Test split is identical across all 3 models (set during Phase 1)
splits_dir = Path(configs['pconv_unet']['data']['splits_dir'])
test_df = pd.read_csv(splits_dir / configs['pconv_unet']['data']['test_csv'])
print(f'Test images: {len(test_df):,}')

MASK_ROOT = EVAL_ROOT / 'test_masks'
MASK_ROOT.mkdir(parents=True, exist_ok=True)

DAMAGE_TYPES = ['brush', 'crack', 'paint_loss', 'stain']
DIFFICULTIES = ['light', 'medium', 'heavy']
IMAGE_SIZE = configs['pconv_unet']['data']['image_size']

BASE_SEED = 12345

# Map damage-type names → MaskGenerator methods (generates ONE damage type)
DAMAGE_METHODS = {
    'brush':      'random_brush_strokes',
    'crack':      'simulated_cracks',
    'paint_loss': 'simulated_paint_loss',
    'stain':      'random_aging_stains',
}

def mask_path(img_id, damage, difficulty):
    return MASK_ROOT / damage / difficulty / f'{img_id:05d}.png'

def generate_single_damage_mask(seed, h, w, damage, difficulty):
    """Generate a binary mask containing exactly ONE damage type at the given
    difficulty.  Uses MaskGenerator's per-type primitive then post-hoc
    adjusts to hit the midpoint of the difficulty's hole-ratio range."""
    rng = np.random.default_rng(seed)
    gen = MaskGenerator(seed=seed)

    lo, hi = DIFFICULTY_RANGES[difficulty]
    target_ratio = float(rng.uniform(lo, hi))
    target_holes = int(target_ratio * h * w)

    # Initial draw at a density that roughly reaches target_ratio
    density = float(np.clip(target_ratio * 2.5, 0.05, 1.0))
    method = getattr(gen, DAMAGE_METHODS[damage])
    mask = method(h, w, density).astype(np.float32)

    # Post-hoc correction to exactly target_holes (same trick as MaskGenerator.generate)
    flat = mask.reshape(-1)
    cur_holes = int((flat == 0).sum())
    if cur_holes < target_holes:
        valid_idx = np.flatnonzero(flat == 1.0)
        n = min(target_holes - cur_holes, valid_idx.size)
        if n > 0:
            chosen = rng.choice(valid_idx, size=n, replace=False)
            flat[chosen] = 0.0
    elif cur_holes > target_holes:
        hole_idx = np.flatnonzero(flat == 0.0)
        n = min(cur_holes - target_holes, hole_idx.size)
        if n > 0:
            chosen = rng.choice(hole_idx, size=n, replace=False)
            flat[chosen] = 1.0
    return mask

for damage in DAMAGE_TYPES:
    for diff in DIFFICULTIES:
        (MASK_ROOT / damage / diff).mkdir(parents=True, exist_ok=True)

n_existing = sum(1 for _ in MASK_ROOT.rglob('*.png'))
n_target = len(test_df) * len(DAMAGE_TYPES) * len(DIFFICULTIES)
print(f'Existing masks on disk: {n_existing:,} / {n_target:,}')

if n_existing < n_target:
    pbar = tqdm(total=n_target, desc='Generating masks', leave=False)
    pbar.update(n_existing)
    for img_id in range(len(test_df)):
        for d_i, damage in enumerate(DAMAGE_TYPES):
            for f_i, diff in enumerate(DIFFICULTIES):
                out = mask_path(img_id, damage, diff)
                if out.exists():
                    continue
                # Deterministic per-cell seed: BASE_SEED + img_id*100 + d_i*10 + f_i
                seed = BASE_SEED + img_id * 100 + d_i * 10 + f_i
                mask = generate_single_damage_mask(seed, IMAGE_SIZE, IMAGE_SIZE, damage, diff)
                Image.fromarray((mask * 255).astype(np.uint8)).save(out)
                pbar.update(1)
    pbar.close()

# Hash a sample of masks to verify determinism on resume
sample_files = sorted(MASK_ROOT.rglob('*.png'))[:64]
h = hashlib.md5()
for f in sample_files:
    h.update(f.read_bytes())
print(f'\nMask cache hash (first 64 files): {h.hexdigest()[:16]}…')
print(f'Total masks: {sum(1 for _ in MASK_ROOT.rglob("*.png")):,}')

## Cell 5 — Inference + Per-image Metrics

For each model: load `best.pth`, iterate (test image, mask) pairs, run
inference, compute **per-image PSNR / SSIM / LPIPS** on the composited
output (PConv-style: hole pixels from model output, valid pixels from GT).

Per-image metrics go to `per_image_metrics.csv` — the master artefact
every downstream table and figure derives from.  Resume-safe: if the CSV
already has a model's rows, skip that model.

In [ ]:
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import (
    peak_signal_noise_ratio as fn_psnr,
    structural_similarity_index_measure as fn_ssim,
)
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

from src.models.registry import build_model
from src.utils.checkpoint import load_checkpoint

device = torch.device('cuda')
PER_IMG_CSV = EVAL_ROOT / 'per_image_metrics.csv'

# LPIPS instance shared across all models / batches
lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True).to(device)
lpips_metric.eval()

def load_image_to_tensor(path):
    img = Image.open(path).convert('RGB')
    # Aspect-preserving resize + center crop to IMAGE_SIZE
    scale = IMAGE_SIZE / min(img.width, img.height)
    nw, nh = int(img.width * scale), int(img.height * scale)
    img = img.resize((nw, nh), Image.LANCZOS)
    left = (nw - IMAGE_SIZE) // 2
    top  = (nh - IMAGE_SIZE) // 2
    img = img.crop((left, top, left + IMAGE_SIZE, top + IMAGE_SIZE))
    arr = np.asarray(img, dtype=np.float32) / 255.0          # [0, 1]
    return torch.from_numpy(arr).permute(2, 0, 1)            # (3, H, W)

def load_mask_to_tensor(path):
    arr = np.asarray(Image.open(path), dtype=np.float32) / 255.0  # binary [0,1]
    arr = (arr > 0.5).astype(np.float32)
    return torch.from_numpy(arr).unsqueeze(0)                # (1, H, W)

# Resume: read existing CSV and skip already-evaluated models
done_models = set()
if PER_IMG_CSV.exists():
    df_prev = pd.read_csv(PER_IMG_CSV)
    done_models = set(df_prev['model'].unique())
    print(f'Resuming — already evaluated: {sorted(done_models)}')

rows_all = []
if PER_IMG_CSV.exists():
    rows_all = df_prev.to_dict('records')

for arch in MODELS:
    if arch in done_models:
        print(f'\n[{arch}] already done — skipping.')
        continue

    print(f'\n[{arch}] loading checkpoint...')
    cfg = configs[arch]
    model = build_model(cfg).to(device)
    ckpt = Path(cfg['checkpoint']['dir']) / 'best.pth'
    load_checkpoint(ckpt, model=model, map_location=str(device))
    model.eval()

    rows_model = []
    for img_id in tqdm(range(len(test_df)), desc=f'{arch}', leave=False):
        img_path = test_df.iloc[img_id]['path']
        gt = load_image_to_tensor(img_path).to(device)         # (3, H, W) in [0, 1]

        for damage in DAMAGE_TYPES:
            for diff in DIFFICULTIES:
                mask = load_mask_to_tensor(mask_path(img_id, damage, diff)).to(device)
                masked = gt * mask                              # (3, H, W)

                with torch.no_grad():
                    output, _ = model(masked.unsqueeze(0), mask.unsqueeze(0))
                    output = output.clamp(0.0, 1.0)
                    comp = output * (1.0 - mask) + gt * mask    # composited

                p_val = float(fn_psnr(comp, gt.unsqueeze(0), data_range=1.0).item())
                s_val = float(fn_ssim(comp, gt.unsqueeze(0), data_range=1.0).item())
                l_val = float(lpips_metric(comp, gt.unsqueeze(0)).item())

                rows_model.append({
                    'model': arch,
                    'img_id': img_id,
                    'damage': damage,
                    'difficulty': diff,
                    'psnr': p_val,
                    'ssim': s_val,
                    'lpips': l_val,
                })

    rows_all.extend(rows_model)
    pd.DataFrame(rows_all).to_csv(PER_IMG_CSV, index=False)
    print(f'[{arch}] done — appended {len(rows_model):,} rows  →  {PER_IMG_CSV}')

    del model
    torch.cuda.empty_cache()

df_pi = pd.read_csv(PER_IMG_CSV)
print(f'\nFinal per-image metrics: {len(df_pi):,} rows  '
      f'({len(df_pi)//len(MODELS):,} per model)')
print(df_pi.groupby('model')[['psnr', 'ssim', 'lpips']].mean().round(4))

## Cell 6 — FID per (model, difficulty)

FID is set-level (cannot be averaged from per-image scores).  Run a
second pass to populate FID accumulators — one per (model, difficulty)
cell — and write to `fid_results.csv`.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

FID_CSV = EVAL_ROOT / 'fid_results.csv'

fid_rows = []
if FID_CSV.exists():
    fid_rows = pd.read_csv(FID_CSV).to_dict('records')
    print(f'Resuming FID — already computed cells: {len(fid_rows)}')

done_cells = {(r['model'], r['difficulty']) for r in fid_rows}

for arch in MODELS:
    if all((arch, d) in done_cells for d in DIFFICULTIES):
        print(f'[{arch}] FID already done — skipping.')
        continue

    print(f'\n[{arch}] loading model for FID...')
    cfg = configs[arch]
    model = build_model(cfg).to(device)
    ckpt = Path(cfg['checkpoint']['dir']) / 'best.pth'
    load_checkpoint(ckpt, model=model, map_location=str(device))
    model.eval()

    for diff in DIFFICULTIES:
        if (arch, diff) in done_cells:
            continue
        print(f'  difficulty={diff}: accumulating FID over {len(test_df) * len(DAMAGE_TYPES):,} pairs...')
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        for img_id in tqdm(range(len(test_df)), desc=f'{arch}/{diff}', leave=False):
            img_path = test_df.iloc[img_id]['path']
            gt = load_image_to_tensor(img_path).to(device).unsqueeze(0)
            for damage in DAMAGE_TYPES:
                mask = load_mask_to_tensor(mask_path(img_id, damage, diff)).to(device).unsqueeze(0)
                with torch.no_grad():
                    output, _ = model(gt * mask, mask)
                    output = output.clamp(0.0, 1.0)
                    comp = output * (1.0 - mask) + gt * mask
                fid.update(gt, real=True)
                fid.update(comp, real=False)

        fid_val = float(fid.compute().item())
        fid_rows.append({'model': arch, 'difficulty': diff, 'fid': fid_val})
        pd.DataFrame(fid_rows).to_csv(FID_CSV, index=False)
        print(f'    FID = {fid_val:.3f}')
        del fid
        torch.cuda.empty_cache()

    del model
    torch.cuda.empty_cache()

df_fid = pd.read_csv(FID_CSV)
print('\nFID results')
print(df_fid.pivot(index='model', columns='difficulty', values='fid').round(2))

## Cell 7 — Statistical Tests

Paired Wilcoxon signed-rank test on per-image scores: PConv vs each
baseline, for each metric (PSNR, SSIM, LPIPS).  6 main tests total →
Bonferroni α = 0.05 / 6 = 0.00833.

Pairing: PConv's score on (img_id=i, damage=d, difficulty=f) is paired
with the baseline's score on the **same** (i, d, f) — the pre-computed
test mask cache guarantees identical inputs.

Effect size: median of per-image differences with bootstrap 95 % CI.

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

df_pi = pd.read_csv(EVAL_ROOT / 'per_image_metrics.csv')

# Pivot to wide form: one row per (img_id, damage, difficulty), columns per model
wide = df_pi.pivot_table(
    index=['img_id', 'damage', 'difficulty'],
    columns='model',
    values=['psnr', 'ssim', 'lpips'],
).reset_index()
wide.columns = ['_'.join(c).strip('_') if isinstance(c, tuple) else c for c in wide.columns]

BASELINES = ['unet_baseline', 'gated_unet']
ALPHA_BONF = 0.05 / (len(BASELINES) * 3)            # 6 main tests

def bootstrap_median_ci(differences, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    medians = np.empty(n_boot)
    n = len(differences)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        medians[i] = np.median(differences[idx])
    return float(np.median(differences)), float(np.percentile(medians, 2.5)), float(np.percentile(medians, 97.5))

stats_rows = []
for baseline in BASELINES:
    for metric in ['psnr', 'ssim', 'lpips']:
        col_pconv = f'{metric}_pconv_unet'
        col_base  = f'{metric}_{baseline}'
        # PConv expected to be better → higher PSNR/SSIM, lower LPIPS
        alt = 'greater' if metric in ('psnr', 'ssim') else 'less'
        if metric in ('psnr', 'ssim'):
            diff = wide[col_pconv].to_numpy() - wide[col_base].to_numpy()    # positive = PConv better
        else:
            diff = wide[col_base].to_numpy() - wide[col_pconv].to_numpy()    # positive = PConv better (lower LPIPS)

        stat, p = wilcoxon(wide[col_pconv].to_numpy(), wide[col_base].to_numpy(), alternative=alt)
        m, lo, hi = bootstrap_median_ci(diff)
        stats_rows.append({
            'baseline': baseline,
            'metric': metric,
            'wilcoxon_stat': float(stat),
            'p_value': float(p),
            'p_significant_bonferroni': bool(p < ALPHA_BONF),
            'effect_median': m,
            'effect_ci_lo': lo,
            'effect_ci_hi': hi,
            'n_pairs': len(diff),
        })

df_stats = pd.DataFrame(stats_rows)
df_stats.to_csv(EVAL_ROOT / 'stats_results.csv', index=False)

print(f'Bonferroni-corrected α = {ALPHA_BONF:.5f}  (over {len(BASELINES) * 3} tests)\n')
print(df_stats.to_string(index=False, float_format=lambda x: f'{x:.4g}'))

## Cell 8 — LaTeX Tables

Generate Tables 1 and 2 (overall and per-difficulty) ready to paste
into the thesis.

In [ ]:
TABLES_DIR = EVAL_ROOT / 'tables'
TABLES_DIR.mkdir(exist_ok=True)

# ── Table 1: Overall comparison ─────────────────────────────────────────
params_count = {}
for arch in MODELS:
    cfg = configs[arch]
    m = build_model(cfg)
    params_count[arch] = sum(p.numel() for p in m.parameters())
    del m

agg = df_pi.groupby('model')[['psnr', 'ssim', 'lpips']].agg(['mean', 'std']).round(4)
fid_overall = df_fid.groupby('model')['fid'].mean().round(2)

t1 = pd.DataFrame({
    'Params (M)': [params_count[a] / 1e6 for a in MODELS],
    'PSNR ↑':    [f"{agg.loc[a, ('psnr', 'mean')]:.3f} ± {agg.loc[a, ('psnr', 'std')]:.3f}" for a in MODELS],
    'SSIM ↑':    [f"{agg.loc[a, ('ssim', 'mean')]:.4f} ± {agg.loc[a, ('ssim', 'std')]:.4f}" for a in MODELS],
    'LPIPS ↓':   [f"{agg.loc[a, ('lpips', 'mean')]:.4f} ± {agg.loc[a, ('lpips', 'std')]:.4f}" for a in MODELS],
    'FID ↓':     [f"{fid_overall.get(a, float('nan')):.2f}" for a in MODELS],
}, index=['PConv U-Net', 'Vanilla U-Net', 'Gated U-Net'])

# Add p-values column from stats
p_psnr = {r['baseline']: r['p_value'] for r in stats_rows if r['metric'] == 'psnr'}
t1['p (vs PConv)'] = ['—', f"{p_psnr['unet_baseline']:.2e}", f"{p_psnr['gated_unet']:.2e}"]

print('Table 1 — Overall comparison\n')
print(t1.to_string())
with open(TABLES_DIR / 'table1_overall.tex', 'w') as f:
    f.write(t1.to_latex(escape=False))
print(f'\nSaved → {TABLES_DIR / "table1_overall.tex"}')

# ── Table 2: Per-difficulty breakdown ───────────────────────────────────
t2_data = []
for arch in MODELS:
    for diff in DIFFICULTIES:
        sub = df_pi[(df_pi['model'] == arch) & (df_pi['difficulty'] == diff)]
        fid_v = df_fid[(df_fid['model'] == arch) & (df_fid['difficulty'] == diff)]['fid'].values
        t2_data.append({
            'Model': arch,
            'Difficulty': diff,
            'PSNR ↑': f"{sub['psnr'].mean():.3f}",
            'SSIM ↑': f"{sub['ssim'].mean():.4f}",
            'LPIPS ↓': f"{sub['lpips'].mean():.4f}",
            'FID ↓': f"{fid_v[0]:.2f}" if len(fid_v) else '—',
        })
t2 = pd.DataFrame(t2_data)
print('\nTable 2 — Per-difficulty breakdown\n')
print(t2.to_string(index=False))
with open(TABLES_DIR / 'table2_per_difficulty.tex', 'w') as f:
    f.write(t2.to_latex(index=False, escape=False))
print(f'\nSaved → {TABLES_DIR / "table2_per_difficulty.tex"}')

## Cell 9 — Figures

Three thesis figures:
* **Figure 1**: Bar chart of PSNR per difficulty with bootstrap 95 % CI error bars.
* **Figure 2**: 6×5 qualitative grid (6 hand-picked test images × {input, PConv, U-Net, Gated, GT}).
* **Figure 3**: Box-plot of per-image PSNR differences (PConv − Baseline) per damage type.

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = EVAL_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# ── Figure 1: PSNR per difficulty bar chart ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(DIFFICULTIES))
width = 0.27
colors = {'pconv_unet': '#2c7fb8', 'unet_baseline': '#7fcdbb', 'gated_unet': '#fdae6b'}

for i, arch in enumerate(MODELS):
    means, los, his = [], [], []
    for diff in DIFFICULTIES:
        sub = df_pi[(df_pi['model'] == arch) & (df_pi['difficulty'] == diff)]['psnr'].to_numpy()
        m, lo, hi = bootstrap_median_ci(sub, n_boot=2000)
        means.append(m); los.append(m - lo); his.append(hi - m)
    ax.bar(x + (i - 1) * width, means, width,
           label=arch.replace('_', ' '), color=colors[arch],
           yerr=[los, his], capsize=4, edgecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(DIFFICULTIES)
ax.set_ylabel('PSNR (dB)')
ax.set_xlabel('Mask difficulty')
ax.set_title('Test-set PSNR by difficulty (bootstrap 95 % CI)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_psnr_per_difficulty.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig1_psnr_per_difficulty.pdf', bbox_inches='tight')
plt.show()

# ── Figure 3: PSNR-difference box-plot (cheap, do before figure 2) ──────
fig, ax = plt.subplots(figsize=(10, 5))
wide_dam = df_pi.pivot_table(
    index=['img_id', 'damage', 'difficulty'],
    columns='model', values='psnr',
).reset_index()
data = []; labels = []
for damage in DAMAGE_TYPES:
    for baseline, color in zip(BASELINES, ['#7fcdbb', '#fdae6b']):
        sub = wide_dam[wide_dam['damage'] == damage]
        diff = sub['pconv_unet'].to_numpy() - sub[baseline].to_numpy()
        data.append(diff)
        labels.append(f'{damage}\n{baseline.replace("_", " ")}')

bp = ax.boxplot(data, labels=labels, showmeans=True, patch_artist=True)
for patch, baseline_idx in zip(bp['boxes'], list(range(len(data)))):
    patch.set_facecolor('#7fcdbb' if baseline_idx % 2 == 0 else '#fdae6b')
ax.axhline(0, color='red', linestyle='--', alpha=0.6)
ax.set_ylabel('PSNR difference (PConv − Baseline) (dB)')
ax.set_title('Per-image PSNR difference by damage type\n(positive = PConv wins)')
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_psnr_difference_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Figure 2: qualitative grid ──────────────────────────────────────────
# Pick 6 test images: one per damage type plus 2 random
rng = np.random.default_rng(42)
picks = []
# 4 across damage types at medium difficulty
for damage in DAMAGE_TYPES:
    subset = df_pi[(df_pi['model'] == 'pconv_unet') & (df_pi['damage'] == damage) &
                   (df_pi['difficulty'] == 'medium')]
    # Median-PSNR sample (representative)
    median_idx = (subset['psnr'] - subset['psnr'].median()).abs().idxmin()
    picks.append((int(subset.loc[median_idx, 'img_id']), damage, 'medium'))
# 2 hard cases (heavy difficulty, paint loss) where PConv struggled
subset = df_pi[(df_pi['model'] == 'pconv_unet') & (df_pi['difficulty'] == 'heavy') &
               (df_pi['damage'] == 'paint_loss')].sort_values('psnr').head(2)
for _, r in subset.iterrows():
    picks.append((int(r['img_id']), 'paint_loss', 'heavy'))

print(f'Selected {len(picks)} test images for qualitative grid.')

# Render the grid
models_loaded = {}
for arch in MODELS:
    cfg = configs[arch]
    model = build_model(cfg).to(device)
    load_checkpoint(Path(cfg['checkpoint']['dir']) / 'best.pth', model=model, map_location=str(device))
    model.eval()
    models_loaded[arch] = model

n_rows = len(picks)
n_cols = 5  # input | PConv | Vanilla | Gated | GT
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
if n_rows == 1:
    axes = axes[None, :]

col_titles = ['Damaged input', 'PConv U-Net', 'Vanilla U-Net', 'Gated U-Net', 'Ground truth']
for c, t in enumerate(col_titles):
    axes[0, c].set_title(t, fontweight='bold', fontsize=10)

for r, (img_id, damage, diff) in enumerate(picks):
    img_path = test_df.iloc[img_id]['path']
    gt = load_image_to_tensor(img_path).to(device).unsqueeze(0)
    mask = load_mask_to_tensor(mask_path(img_id, damage, diff)).to(device).unsqueeze(0)
    masked = gt * mask
    axes[r, 0].imshow(masked[0].cpu().permute(1, 2, 0).numpy())
    for c, arch in enumerate(MODELS, start=1):
        with torch.no_grad():
            output, _ = models_loaded[arch](masked, mask)
            comp = output.clamp(0, 1) * (1 - mask) + gt * mask
        axes[r, c].imshow(comp[0].cpu().permute(1, 2, 0).numpy())
    axes[r, 4].imshow(gt[0].cpu().permute(1, 2, 0).numpy())
    for c in range(n_cols):
        axes[r, c].axis('off')
    axes[r, 0].set_ylabel(f'{damage}\n{diff}', rotation=0, ha='right', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_qualitative_grid.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig2_qualitative_grid.pdf', bbox_inches='tight')
plt.show()

for arch in models_loaded:
    del models_loaded[arch]
torch.cuda.empty_cache()

print(f'\nAll figures saved to {FIG_DIR}')
print(f'All tables saved to {TABLES_DIR}')
print(f'All CSVs saved to {EVAL_ROOT}')